# 09 Deploy Production

This notebook packages the final production-ready model bundle after successful registry promotion.

Outputs:
- `dvc_pipeline/deploy/model.joblib`
- `dvc_pipeline/deploy/feature_columns.json`
- `dvc_pipeline/deploy/deployment_manifest.json`


## Research Paper Alignment

Deployment remains tied to promotion status so only validated models are packaged for serving.


In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys
import yaml

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "params.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

def read_json(path: str):
    return json.loads(Path(path).read_text(encoding="utf-8"))

params = yaml.safe_load(Path("params.yaml").read_text(encoding="utf-8"))


In [2]:
params["deployment"]


{'output_dir': 'dvc_pipeline/deploy',
 'deployed_model_path': 'dvc_pipeline/deploy/model.joblib',
 'deployed_feature_columns_path': 'dvc_pipeline/deploy/feature_columns.json',
 'manifest_path': 'dvc_pipeline/deploy/deployment_manifest.json'}

In [3]:
subprocess.run([sys.executable, "dvc_pipeline/src/deploy_production_model.py"], check=True)


CompletedProcess(args=['c:\\aditi\\.venv\\Scripts\\python.exe', 'dvc_pipeline/src/deploy_production_model.py'], returncode=0)

In [4]:
manifest = read_json(params["deployment"]["manifest_path"])
deploy_dir = sorted(p.name for p in Path(params["deployment"]["output_dir"]).iterdir())
manifest, deploy_dir


({'deployed_at_utc': '2026-03-11T06:41:38.104531+00:00',
  'model_name': 'IronOrePriceModel',
  'model_version': '4',
  'source_model_path': 'dvc_pipeline\\models\\final_model.joblib',
  'deployed_model_path': 'dvc_pipeline\\deploy\\model.joblib',
  'source_feature_columns_path': 'dvc_pipeline\\data\\features\\selected_feature_columns.json',
  'deployed_feature_columns_path': 'dvc_pipeline\\deploy\\feature_columns.json',
  'production_model_uri': 'models:/IronOrePriceModel@production',
  'promotion_reason': 'passed_testing_and_staging'},
 ['deployment_manifest.json', 'feature_columns.json', 'model.joblib'])